# 12 — Bures–Wasserstein

For one special family — the Gaussians — optimal transport needs no algorithm at all. $W_2$ is a closed-form expression in the means and covariances.

**Prerequisites:** `11_amari_bridge`.

**What you'll be able to do**
- Compute $W_2$ between Gaussians in closed form (1-D and multivariate) and check it against the LP.
- Read the formula as a translation cost (means) plus a Bures matrix cost (covariances).
- See the two covariance ellipses whose distance you computed.

For the Gaussians, optimal transport needs no solver. The Wasserstein-2 distance between two normal distributions is an explicit expression in their means and covariance matrices — the simplest place where $W_2$ is fully closed-form. And, as the next notebook reveals, the matrix part of that formula is the doorway into quantum optimal transport.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ot

from qot_course import viz
from qot_course.transport.gaussian import bures_wasserstein_distance, bures_matrix_distance

np.random.seed(42)
viz.use_course_style()

## 1. The simplest case: 1-D Gaussians

For two normal distributions on the line, $\mathcal{N}(m_0, \sigma_0^2)$ and $\mathcal{N}(m_1, \sigma_1^2)$, the Wasserstein-2 distance has an elementary closed form:

$$ W_2\big(\mathcal{N}(m_0, \sigma_0^2),\, \mathcal{N}(m_1, \sigma_1^2)\big) = \sqrt{(m_0 - m_1)^2 + (\sigma_0 - \sigma_1)^2}. $$

The two contributions separate cleanly: a **translation cost** $|\Delta m|$ and a **rescaling cost** $|\Delta\sigma|$. So $W_2$ is the ordinary Euclidean distance in the $(m, \sigma)$ plane — the geometry on Gaussians is *flat* in those coordinates. Let's confirm it against the LP.

In [ ]:
m_0, sigma_0 = 0.0, 1.0
m_1, sigma_1 = 4.0, 2.0

w2_closed  = bures_wasserstein_distance([m_0], [[sigma_0 ** 2]], [m_1], [[sigma_1 ** 2]])
w2_formula = float(np.sqrt((m_0 - m_1) ** 2 + (sigma_0 - sigma_1) ** 2))

# Cross-check via a grid-discretised LP.
grid = np.linspace(-5.0, 12.0, 400)
def gaussian_1d(x, m, s):
    return np.exp(-0.5 * ((x - m) / s) ** 2) / (s * np.sqrt(2 * np.pi))
mass_0 = gaussian_1d(grid, m_0, sigma_0); mass_0 /= mass_0.sum()
mass_1 = gaussian_1d(grid, m_1, sigma_1); mass_1 /= mass_1.sum()
cost_grid = (grid.reshape(-1, 1) - grid.reshape(1, -1)) ** 2
w2_lp = float(np.sqrt(ot.emd2(mass_0, mass_1, cost_grid)))

print(f"W2 via Bures-Wasserstein code = {w2_closed:.6f}")
print(f"W2 via 1-D formula            = {w2_formula:.6f}")
print(f"W2 via grid-discretised LP    = {w2_lp:.6f}")

**Read the output.** All three numbers agree: the closed form, the library implementation, and the brute-force LP. The Gaussian world is the simplest place where Wasserstein is fully explicit — and the formula generalises cleanly to many dimensions.

## 2. Multivariate Gaussians — the Bures–Wasserstein formula

For two Gaussians $\mathcal{N}(m_0, \Sigma_0)$ and $\mathcal{N}(m_1, \Sigma_1)$ on $\mathbb{R}^d$ (Dowson & Landau, 1982; Olkin & Pukelsheim, 1982):

$$ W_2^2 = \|m_0 - m_1\|^2 + \mathrm{tr}(\Sigma_0) + \mathrm{tr}(\Sigma_1) - 2\,\mathrm{tr}\sqrt{\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}}. $$

The first term is the **squared mean shift**; the second — the **Bures matrix distance** — measures how the covariances differ in shape and orientation. Let's verify it on a 2-D example with different positions *and* different covariance shapes.

In [ ]:
mean_0 = np.array([-2.5, -1.0])
cov_0  = np.array([[1.5, 0.4], [0.4, 0.6]])     # elongated, tilted one way
mean_1 = np.array([2.5, 1.5])
cov_1  = np.array([[0.5, -0.3], [-0.3, 1.4]])   # elongated, tilted the other way

w2          = bures_wasserstein_distance(mean_0, cov_0, mean_1, cov_1)
mean_term   = float(np.sum((mean_0 - mean_1) ** 2))
matrix_term = bures_matrix_distance(cov_0, cov_1)
print(f"||m0 - m1||^2                            = {mean_term:.4f}")
print(f"bures_matrix_distance(Sigma_0, Sigma_1)   = {matrix_term:.4f}")
print(f"W2 (closed form)                          = {w2:.4f}")

# Cross-check via a grid LP on a 20x20 mesh.
g = np.linspace(-7.0, 7.0, 20)
X, Y = np.meshgrid(g, g)
pts = np.stack([X.ravel(), Y.ravel()], axis=-1)
def gaussian_2d(pts, mean, cov):
    diff = pts - mean
    return np.exp(-0.5 * np.einsum("ij,jk,ik->i", diff, np.linalg.inv(cov), diff))
p0 = gaussian_2d(pts, mean_0, cov_0); p0 /= p0.sum()
p1 = gaussian_2d(pts, mean_1, cov_1); p1 /= p1.sum()
cost_2d = np.sum((pts[:, None, :] - pts[None, :, :]) ** 2, axis=-1)
w2_lp = float(np.sqrt(ot.emd2(p0, p1, cost_2d)))
print(f"\nW2 (grid-LP cross-check, 20x20 mesh)      = {w2_lp:.4f}")
print(f"agreement within grid resolution?          {bool(np.isclose(w2, w2_lp, rtol=0.05))}")

**Read the output.** The closed form and the grid LP agree within the discretisation error of the coarse $20 \times 20$ mesh. The **mean term** is the cost of moving the centre of mass; the **matrix term** is the cost of reshaping the covariance. That matrix term is about to become the most important object of the whole course — but first, let's picture the two Gaussians behind these numbers.

In [ ]:
def ellipse(mean, cov, n_std=2.0, **kwargs):
    """A covariance ellipse at n_std standard deviations."""
    vals, vecs = np.linalg.eigh(0.5 * (cov + cov.T))
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = float(np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0])))
    width, height = (2.0 * n_std * np.sqrt(np.clip(vals, 0.0, None))).tolist()
    return patches.Ellipse(xy=tuple(mean), width=width, height=height, angle=angle, **kwargs)

fig, ax = plt.subplots(figsize=(7, 6))
ax.add_patch(ellipse(mean_0, cov_0, edgecolor=viz.SOURCE_COLOR, facecolor=viz.SOURCE_COLOR, alpha=0.35, lw=2, label="source"))
ax.add_patch(ellipse(mean_1, cov_1, edgecolor=viz.TARGET_COLOR, facecolor=viz.TARGET_COLOR, alpha=0.35, lw=2, label="target"))
ax.scatter(*mean_0, color=viz.SOURCE_COLOR, s=60, zorder=3)
ax.scatter(*mean_1, color=viz.TARGET_COLOR, s=60, zorder=3)
ax.set_xlim(-6, 6)
ax.set_ylim(-5, 5)
ax.set_aspect("equal")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("Two 2-D Gaussians: W2 moves the centre and reshapes the ellipse", pad=12)
ax.legend(loc="upper left")
plt.show()

**Read the figure.** The periwinkle and amber ellipses are the two Gaussians — different centres and different shapes/orientations. $W_2$ bundles both differences into one number: the mean term accounts for sliding the centre from one dot to the other, and the matrix term for rotating and stretching the source ellipse until it matches the target. The next notebook asks what that matrix term *is*.

## Your turn

1. **The flat $(m, \sigma)$ plane.** Take a grid of 1-D Gaussians with varying $(m, \sigma)$, build their pairwise $W_2$ matrix from the formula, and embed it with `sklearn.manifold.MDS`. Confirm it lays out flat — it *is* the $(m, \sigma)$ half-plane.
2. **Mean-only versus covariance-only.** Build two Gaussians with the same mean but different covariances. Verify $W_2$ reduces to $\sqrt{\texttt{bures\_matrix\_distance}(\Sigma_0, \Sigma_1)}$ — the pure shape cost, with no translation.
3. **Scale of the mesh.** Re-run the 2-D cross-check with a $40 \times 40$ mesh instead of $20 \times 20$. Does the LP value move closer to the closed form? What does that say about the discretisation error?

## What you built

- You computed $W_2$ between Gaussians in closed form, in 1-D and multivariate, and checked both against the LP.
- You read the formula as a translation cost (means) plus a Bures matrix cost (covariances).
- You pictured the two covariance ellipses whose distance you measured.

You now have optimal transport with no solver in the loop — pure linear algebra on means and covariances. That clean formula is about to reveal a secret.

## What's next

In `13_the_bures_bridge` we look hard at the matrix term and find it is the **Bures distance** — and for unit-trace matrices, it is *exactly* the quantum Bures distance of `02/12`. Replace covariance by density matrix and this formula defines a quantum Wasserstein. That is the doorway to Module 4.

## References

- D. C. Dowson & B. V. Landau, "The Fréchet distance between multivariate normal distributions", *J. Multivar. Anal.* 12, 450–455 (1982).
- I. Olkin & F. Pukelsheim, "The distance between two random vectors with given dispersion matrices", *Linear Algebra Appl.* 48, 257–263 (1982).
- R. Bhatia, T. Jain, Y. Lim, "On the Bures–Wasserstein distance between positive definite matrices", *Expo. Math.* 37, 165–191 (2019). DOI:10.1016/j.exmath.2018.01.002

**Previous:** `notebooks/03_ClassicalOptimalTransport/11_amari_bridge.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/13_the_bures_bridge.ipynb`